# VidBest GPU shots
Select GPU, then Run all. The included CPU episode supplies the reference frames and original sound. Only the selected shot windows are regenerated. Choose ltx, wan or toon below. GPU capacity and model downloads depend on your account. Keep the output before disconnecting.

In [ ]:
MODEL = "ltx"  # ltx, wan, toon
USE_R2 = False  # Optional notebook secrets for checkpoints


In [ ]:
import base64, io, json, os, subprocess, sys, shutil, tempfile
from pathlib import Path
from zipfile import ZipFile
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select GPU in notebook settings, reconnect, then rerun.")
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30,1), "GiB VRAM")
temporary=Path(tempfile.gettempdir())/"vidbest-gpu"
temporary.mkdir(exist_ok=True)
studio=temporary/"worker"
root=temporary/"ComfyUI"
output=Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
bundle=temporary/"episode.zip"
ZipFile(io.BytesIO(base64.b64decode('UEsDBBQAAAAIANSJb1179CXxYAYAAFkNAAAIAAAAc2V0dXAucHmVVltv2zgWfvev0PJJRiVZN9tyAD2kaboNkE6KaabdQRAQFEnZnEiihqTcBIv973tIyY6Tpg+jB1sUz/U73zkkQuiq04Y0jSc77mnecGo48y5kWz/9ceXtBePSqwh94B3zRGekR7xOGl5J+eBJ5V2Lbnj01NB1XEUIoZloe6mMR9S2J0rzw1rqWa1k6/XE7BpRedPnL7A8iOjdYERzXA1VryTlWh+/POnZ7OLm88c/vdJDnPPlJimWdZ3XeUZ4yjaEJWlapflylWxYumYbRghDs9ubm9+sBsuSlK7STZHWybJIq/VmuczznGUkTQnd5EVdbegqztDs+7lTcCCEN2q7+E46nEYJnmDBivcACdly5oTx75ffrEKSZoTWCaXrvKK0qjZVXdcFoQnn65QzGhc0zfIElD5eXV9+BZX/zjx4UGMe0Zl35xb28dG12O6MEvRBL65v/xN+s3VAASo2RV6TdBnH6zpbr2hC0s0mTlYs3izXJFuTtF5v1lUFomAzdOUL0yrcx9EmWkaa1NzwTkulQYLuOH3oJVRVo3lw4vw58VbSncC94nvBf5ymHSDO8qpYFktGAcI0zrPNOt0kNFvyJI6LIl8Vq2RdsBWI6r4RBtei4Xph+KPBvKOScQWr5eNjg+s+Wb2K7YUYmt+P4aEfpLNA+QB6MAH/0jx6JwxvA/sTue8+WqD5XXw/92qgq/0ONPb8Y7aIiboetJAdbsFZoxfgw9ZapHucF3GPk/w9RFhgTUnD2as494RbBccOeH+5++zkZdZDa5Z4TLzAPG+zunvbOm1Ej/fCRrc4ece7F3JHdIyU3SseQW9XAmB4ImKhWQghhrUJW81DqcRWdKQBN9AUVUZYFmcx5dmKJ0u+jumSVdBDCatjtoqhcdb1mO/BQJHH8IS8JWEPA+AtaF7S6jPp5F/VwobU8PCIe5iGSVgRzUEnjjc5y4uMJGlcZ3WWMVJXLCnWnOSErGKWxIyvl/QVQxaudNEbPLKove3R7oQ/aVgs/zebMV57rj3kYHw7swIPyC/t71iC+ZlLTNR2GnpOZIGirTBoHvFHoY32JxH72O2ofWBCgaTi0HDlrRp44CSxfHCr+VH8efZFAKx/h6zdAInO/YV/o8AavA9chCe6MHR1eZAOLyYxt0UHZf2Wr0xbjXd3CLIK3bgGvU+X5x8Q2Ca9GRTHAEA/mDFcC/mJO0h+Mhtpw0AQ/pTo/fm/ygNMZ7/KaXIMwNOdDTZkvDe7MoH3nTG9PlssII3dUEVUttDVFv13I8DBwfjPAPza0aGYky9DnNuPl7cXn/Ah4xNjjgBiPBl9JaUJPEexwNNyUGAcy655Kj+SRvOpzke6OPGTGTodGihwZ9cYqOJ/D0Lx1lHhzmos0Om3yDwadH9A2bkuy6m/j5kSxmRXjsp00Ea2uANJDRPvw1NHWnGhSG24+q5I33OFjorHUJ2FAD2Iv2A8THGGb+oG9hR9BvlFrFaiY6OxN9I4kuUUurMTU8CzbpQZD3ojFd2dtpf7ENGBkUhoTPZENLahTztMEaG59/vQGdHyS6Wk8tFXd5vxSOcdVbzfvl19uDr3/v3lD3tpscJeLZQ20RRmL/ryDm4a0MScDm5uAGNa4ArsuB50nJhqY48UyNeeKKdp/5L1YAO4GAKc0Ck+qMx/5rC7Ju2GLQznbU0ox9AEB2R2tV1hJn90jSTsJAaYTdAWgT0Eg1o2MBNtUO6acefoc/8clIGe4GYiznjoocWodJSZkB+/RlBe/QPa0Ucvh+WzzRO749/CXu18G888Aj7xV96jcQ7+g6FIoWU5K6WOtjaAvY+glO8vv97izzcfLq/xxfnFp0v0LN9KqC8ouDhG5flicq54Q4zYc2yk/wKGuRtqTtiDNfAFrsXP1XQELifLY2eOrwRux9O7pahN25+PFpz/V4XzbcGwYOWhcG6elYcKWrxKB938tCBT8GBfP7WN6B78NwsQDd24+bqY0/bPx9OJ7mTZAjNmC1hp2ewhHZiKYAhjGx3GMI4wbonoMJ5GUl8eLv3RudoOthO+2JWaIukjmBCYTFv+CDkC/ksBDVI6rr4tGYa2SCgwTz139QxgPJOhMWNxEdyO33Nt3ABD81/ZmMZPaMcPCgg1FnK4IEk45QxQbeIOKfvIZWGV9RT74SggkY3kGZOAROO5QKLT6Taf/R9QSwMEFAAAAAgANYpvXWeMz9kOEAAA6SkAAAkAAABoeWJyaWQucHmtOm1v2zjS3/MrtAQOlTayHLtpb89Z3fPstuldsXvboEkPhzMMQZYom43eTpTtZA3/95sZkhJlu9td4BLAlsjhcDjvMzRj7K7hddxw32l4mfIGv+Wm4E5cpk4sJS+WOXf+dvfJkeuqlY4opUhx1uFPQraiXDlvYJLXQlYpDxhjF6Koq6Z14mYFiCU370lVtvypzcXSjKxjubZeP8uqNM+VvMiaqnDquEUQRw/fwasBketNK/LurUoeedu9bZZ1UyVcym7kuXtseVFnIu8Ia0XRPW82IlU7b5ocNg4a/p8Nl60h4KN69WG2qnmpQH8VhM/A/FvU7+DV//f7u+jt7buff3i4fXtx8ffbj7ch0u9GEUJHkQfIZZVvuesFKIKyvfjHh7e3P9+HLsvbJ+azXVzCZ1tVJfMu3t3dh9Pri4uUZ06zKV3gr/RmFw789ccNcGYu28bdek5WNc4WJIaikAs/WfPkMXxoNtxTWGDNkrvIYo0njds4tJDRiqjatPWmdecsy2gFkDTawgdvmqrBF9CMXQR78riQ3TtsXsQtvlYZfKJwmY97LTzaq+CwF44GeRWn0sW91cwWFKwKS9AVV9IZUOsIfs7MLgtHZI6cswSULona55qzRRgyWsoUmoa3m6ZU2PwMNmldhUNTtpizdNPErQDCFpojqViBdG2W7ES7JrVTgwFK3WXNknlgHU6mgPBvHWp1DuQ6nr567Xrd1G6N2rHMQUVnYQZSj1N3cjW9/hY/vB4FoQk2NbCCuwTe49CnWQdr/qSp7KRIBuwuY8n9Oo9Lf7kp05xrxOr85qShkjnCKtxk1bYcEAORGKG1up4Cg022otrI8IpeSSqwEgVDGPpDyDZu2hAHUVrwHGVNXIB4Ooik2pQGguakNQliRWm6tNT7JhQlOINGjdHCfoxev5/iI0F/b4jsRi4J5K8NfKauYcG3YEhHTG9iIbnzzzjf8FtUapd9gncJxsxT3ymrstryJo/rGt0dnbuJS5CB8YbtmpMXJF47cetMr52slsyzj1VWraMODTIoalDAALRZ1O5XibkF6GenqJB6Ry/uUXeisY58cbSnnJ1i/SFNkdScx+DeqrJ38hp3EZciAz0L9wzPMgO2+QwVJ1IKzmZaD0mZwFmJtF2zGfFgrt8WPltzsVq33bh+hQmii83o60A7Kr3VvjAoHlPRuOpFktvyKeRElXFinX0ajx48cHTBcfP8VjQ8aavm2QUk4QAxGW5b1D3X4UV5ZniwrBYxa1+OM2Om0YC7R9/MBg6esMZNshZbPpSnHgx2jWg5emYmE17yQHlEsrx0U9TSNQz3vC+vV1ZOYgiK+poNQdEshW8Mk5cQxxv0JcTjIzUz8CUYoE9miGtcl2WikeC3z1iw50Ngir8weTk06NHEO7Mh6StwOkR+ZmwvDqM9EnAI6nLFzoJTQIPQU9R8hdEkr1Y53/J8EIAg7CBnafMx2jcMCuYrZo0UTTOMWRMEfx4EouO/Ib8RkKADpLNfcaIRGM3BdyYgIuV+lXcGH9rESasH/V3VPNqRxeiXXoNK9GvPtsJ2zL+qwGFrj6UpRpQoxHmvH4vLuWbzo+Kx0hGEIhfm5hDOirk2xoWnkoZHUgWjCUrm3mIoTiL/VxUPiTVIu2wS38UzjmlIR8udjpYpKPdMpW1BUtXPaLLV8rOLq3DOM25LOxZCZOn6N2ExH3igxRm39jYu4hVPHYRzzuQDxcVFAgeSzj24B4DUmQ+IKopEKdoociXPM0t58TVYbjC/DCsZ8HIrGpDKircu+ziNfvz05qfbh6Gvt5YMuaZTxGXVVi8HE7QiyQU4qJBm9QvI+yXzITWvKwh7EeSdFg1z3P/2l7d3H97/8gA+teErCBER8j5k8aY1R6fUptqVqEh0Ov+RP/tWlqPJpnBhka5Z9i7OIY3vNA1TXiQxqRoe8KeE1xiYpDnbG6KbpNF72OZ5Zp0wMMRQJuxaWxJhaMpEXM9TtY2NGxUKHMARezMcw7y6Bor4nBEopnpvIFGErLFUrtFl11fXoNm/VPebZP0Tf6bn9h0mCsxTWjUMyuc4oQcxGHVs3tRfY7LNYJsnaqXmiOGAf8Qb7VWaOnFBFXxK2H+B4K23gGol1CVKP2+FGEqyQXswbXY9MjUYcYQkxiAih8MJ6clfg7vhjYT4/wZLt7IdPWCePWOQCOUioWRqTH7oYMViXRm5QIqPlRVUD+HLK7J/IxajV51zc80MHO//+zox0I8QGMFQGzq55A3kYm5TVa3tR1X5F6gvV/kiTLd7I4a3YCkgD3TZZPrn4Ar+IRhced6Ng2obEgDYNH6jAbnefKLS0jhNgT4ZZmzdtvVsPO7Wz/a49MB6OoiwMdYhS5ABHG7FOi+onCAM9VRhoWUXXHcEOodiFQyLJ5s2hurbZ0UsyqBGJR2NckiBOJaF1ilgGAlRQRCfPIKsdluIfDSdComoRugURnm8KRNMzpJdGiLBsC5FOQFt+AgmZBN1//D2w6cHb2DMJ2lH1IeUyXdXZ0K/oHozCeoqz1H1epXTTvwj5K2gL8qNZ6BzRfb86T22GSALD5x7zp09EEhx+MBOYzeShXahxXXJxsBGyAyhMo0htnk3S4ifjyfLtGf5cE8bz5CEQOac1+50uAcahib1Qem1jjiGUkqHNjX1E1IH5o+IfBY8T4029e5UlHGeH3GUONXypoBJSEKGePCgBLCLResaG5sck6vOZcnRUP1UQ3KcKhSPAqVx06MzDoaaQW4BfiL3SUU03ZTB+CZV9YUPGLW0KTDcvf/ZBIL3GIh9+vxQqwOvmrhe20mNi22Rsdrnko3iWpjM5rQGxRwlNDt3KctcKCttgYvWrKkz/toP6ZJEGau/Dt2X0yv/FfCNik5YrnyfC0M+TKlN2+qRlyG2hQL8uHY9rMBpChURvOOhq4hN/nWcOSF+OmEYqmaO3siAWdZCOOf4uQB/s6fdz+fI5G6Iucq/ULJ0klKTyxHFULuMSKAgSl1RoJsFn4rZzN9+ZJDkA3OA/zLecu3ORFmDNo8t4rxenHP2GgMswaA8GMqMLcJhrWtSuwEferI0pskQk0BCAZXaWLNrcbxo+puLSAQnayZXw0WS81TTrBI7GvCvtVGR7Xc48vYpNOfARt0x8j9/x/CoMKUEzV79NonH58KaLZxeD3BMXlsdlgbS2vC7wfx1Tx0vV1BhKahvi/jJnfiqA+IOS7RvcR+slqBWGytwz7ucHJ/mL6+OTzPgnelXkXWFoDpkeOHa13SoL++ER9MjrC+HLCorgXk+SmEI9xVJGaL/MkSHWRVyO6obnoknQECmBUFCla+qySUgHA4iiNZef2/0eEb4D55R7EikxqNQkw9zwV5RGg458jHONQTwqnkes0vYz6Mz4IOdIqp1s2HAsiLTy65SUpBUircbpcn0hO1YtvgmZHKToPNn54IsZgvHCLz5bPL66kobOLJNhgZGNYJpF81f1GPZGzcWk7TE+356bkP2T2qQke3o7BkiZcZ3vHHadVw67a5ytHr+gfYOFdWU8mHqWybcrt3UiOrvmLejstkXZVYNOybqFMe1G5w2VB5RsYKNcSUwb7PMqhwCJlvoEaNvllXjH5RuuSgfXULlG3LQbZezq1ep8dnASoIIUCoQcWQbpXwbhmbBcFwZxlFd/ZUtehX9He0VkgiyhfnZi30vZOykHsb7oVc5vFC9l25n9ifYlnbFe4MMcGA/MdzjYr+FEASuqqojVIowyTEfpPeuYz1hgxbOcDeYSqixA+XC0/Q11nSjWjxFWYHh93mzvZ5e1brnAzJbmEyCeNUUbcNNjDMSJb/gncR0ctXBFhsN0vVmp4HRCzYlybYQUopy1bcpKaGClSJV3SRFu1YuXXVTh4nam4j9pNocpNyqox+F1vXNcUkqytYlMOUiy2WkGYYFTxiqFxXZVJro9i0U/x7Y/77liv2+lUC+gRyJp3fqjWCPSKWTmktD1arsMkWocmvpKy6bayuegxXzNJy7wq893b6sh5ZI68gkartoWNjcM4gsuz+yBqJFb36jaFa1XbOSoa3/g4st3UJcGF/YYjk8Z/Or2XaBOjy9BjVv61aGdw/3o/uHHz4+wMMc11wtrEzzJAf1TxPVTuM++4obww6uPqE/sZwSEn8J9CCd9cI6+xfy5JvfcTmjTxlAjQ8lANRh8/3ng3XcJM55uN8dZvv1YQbkJjyqGrHCIiaKZQ1ERqQ4YcoTCF94JQUWbha41W4kdt54Ck/rkVjDE3JQxk04OcfKyz1ReRhPr8cPP863QMvCcl6nxCLr959Hk8NCAauLm+eQV1kUJ+oGDBR0Bs4ZVDR8seTtjmMH09c76W/VzX6hDOXFAhB6LzTyjgJzCXsJElCURAnkBDl/ito1Vi/SNJyHszByw4LPlSA/igfAcr2Ia3SNahP0sUbkHmxo5tnVLP4/9rudXjKL4SuOE0JQbbM8XiFRlxkkxXTO3jXW5B3NvTBauOlZpybY+lY5uOIlaaZd+/WdG1MC3kLxft/Gicpi9GCWlG2uHABg5aFuxEIZ6jzy59DcKukuOsJ1FITdU393PoZFY6JsCPtHLpDcbtWYYTbBmwCvX097N8mj/9V0pMOlcpIE2JSs/Z4Xuj2Fj70tE1OCDHdwaRs18POHNz9Ft/+ybqWAOp3MKMTeTRGeu2S4IccZzhfg7qjBBU+mk4UOdJACnbs0sm4FhnkQ6A1Elf6YkFsg4AhKT+rT3zjV8jNUr7oJNt6DgCBNIBHBt1qu2zhDxMDPUNexRxgx4eEhS+JkDdn+cc8XY4AKsAr5UZLg0Y9ZSNm6zrMLNPq44flWlYURYI7xnbl2eF8SOATgomq5Q7+coJa9wy5hqzPtKsDbXRkpsr3jcxkjI/q/dsjTcyx5hvZFhUNRwfqqFIl7SgrspDUDIyy15IymkJYGYEDkvsi8XbsFe4rsa50jXyi2/0+5bhiVOqj155qDZ7h9o5QqY3jxvT/m0khxbxZMsoM80bg/JghLlDOlhvqagJTwjOzxj+zXhLZzQNSUGEDRTYA+FvsALiGn3yUMqVfuwCzZk6WxmfCZrDYQztmM1h+sfm9nlsz8zIuMskv0ju7pQMYq2RvKeSACy9/aOK37VQtCU6wag+peVjUGrZsNBQOVXIo3aFNvUE+fZ/6XPJRNkP9VQusGk+0vgl0IvFik1kMUhiyKsJcfRbr1VYfmV3LBD80KPG/Z3uFbg8EQMm8Z1gGYT4RJOI1LFy9KwREWKfPxt2nYyrWiGf18S9IatcBldYyB7CamwVjv4pK4mI+/q6FwcgqAPwT6TQAVbWyQLodFTqpO6KreQI6RiVLINbNz1xNCcc3v2eQEpOP9b0KNRiRe5ifrSkB6E6rf2vmQ8MSbvFWdvHPL0IdZmLsFFISxo/EjSITuAMx9fAxSo0MhHul2zZo4ALGBEpBEZuZ3W3FAxUkc0G+34M1KfMjETTpmpgCmT8niQHnaOEA6+6zIN5uhALyL/wJQSwMEFAAAAAgA1IlvXRHOvLPrAgAAaAoAAAwAAABsdHgtYXBpLmpzb269VlFvmzAQfu+vQDynUSAkJHubuj5U66RKqapJU4UcfIA1sJltaLqq/31nSICkqZNO055i3/m47+6+u8vLheO4c/eT84IHPDJeVlp1d5Ro2Gi8u9+EZoI7pRRFqd3RTh3nrET1j+0dJdNFp3Wcyfb02Py+tgo0IkpF+rkE8+Wr25u7e/RyzWNBwb3YvnPDc2Dl4sn5VZGc6eeR8ySk0v2VQiJkARRPTGkh9faYsLSS5ly0MakCiOxuRGqWkFirkZNUCqiTMJ6CxOuaUIdwokVhfAGT1MkIx+9Uaf78X1KysKVEkaLMQe27Dv1jrjusNYH990EweO+fAfXh8/UX2Ec5tcI06Yk4KRprPdts8igpvflYkQQ0cIU17HO585LrTd1LKdQsbuRYYlLl2j2Ry1tBKMgeIUZpQ/iz1B1C9HxZMwri0l9f1pPxcjzbg2rznAF+SjCuW/8rZgrUo5gvbSgSiQgiSbT5lh90wZdCMc3qg7otZ9Y6c0jJCSPvjGLf3n9/uBKcMtMp2BaDZvWs1NRQGsF00iEqyCZSGUtMF/vjyaxTrImCToP57jVKS9BxhmItK+g5ArJgnOTNc68T55g5ri0Bn8NuE/AqzoBW+ZA/2FWWaAmlERdMwSHSRhgpAIoab7L0F0EYLINJ4HlBX+A4SU2qxoNcYXvllj59W+/jJEHCfZwke0beG6N26siDqTO1OlIsLcjhoPKsJm01I1aQ1Ebic2q6ahFfVbgTikFNp6eHq+zGAuwI8Z6Xr1s3K8gh1gM31uG4i8+VkIAEHsO4bNvsXYbiaLlprPotYWUnK9IoxiUuQSnsYlQFPdOO5Df86PIyPXMnAf8pxOiih4WFssB6YlSb1g7niw5NBizNTAvPPL9nAvC0ebkMBzMD50Kk2G8DwDvVBX/RBOG/2KK2LC+O7ukTWb4p0nvxYJbTIMuDpXLI+xpapoys9DtozKPl70JJGNbDLCrkU8I2xg9uyzWo3U42wC5e/wBQSwMEFAAAAAgA1IlvXbqXNuscAgAAEggAAA0AAAB0b29uLWFwaS5qc29urVVLa+MwEL73VxifoxK7eXVvbVPYQAsLG3JZFqPEI1cgS0JSsi0l/31HahrXbux42ebgSKP5Zr55Sa8XURQn8bfoFRe43AhqbeZeNKAsflA0X5S0gHjwdsyl3jp7VPeScIy6jBvrLrUs4nC0x+/eo+L0K6wj8KTxqzbjwfAtdZunM9Z98L8OkpCLwXEzPKx+D2qItI5ITyEaNEdtNOfqjxSYhxuZ+3TMXyQt+Z2hzIF5VDmIdvZlOEYbTim5eYNk4yTNuMSVJtpsJeSE6WRyaSkDB9IqY49s4/ydBN06VYmZnmWIdHjCqLDQCGXcM5S7h8WPvhFYR9cCSM4Z21quJElJQjaC68/sG2wmbWy8+yU8u3u5QS/tHBzqeP1H5dBxpI0qtauS4UnUyz3uUe7pV7HCP0Mjako7iEoQjstiEK3F1vwvw+t/qOKK+5r0raVHkBVfku9ni5cM21gssaUPU7AI3awE9fU57/9DHkado+zzlu1CaHXYdSdMK8sd30EdM+nESCjoZ8z0/E1j64irToR1oD0gHVYxsgIF0+MeHMV9Uo26oWVwkkwqMwA5SkZppRU0Kqs7Cln97mjWtfVBWd3c+64C015Ibx0vwaCOa8IcKS2Q2WiIPwIlPVxsXY016/A/h+7Rs7TUopl57NSu1CPPhv7JZ6Q5g61D+JPuoM/L2GA56yTJuACf2EwbYPw55Jfna7DuQ/ou9n8BUEsDBBQAAAAIANSJb11HD8yimQMAANsMAAAMAAAAd2FuLWFwaS5qc29uvVZLj9s2EL7vrzB0DhaU+FRvTbMo0iZFgN1uD91AoKmRTUCWBEvyug3y3zukLcnSGuymCHqxqY/z+ObBIb/crFYRjX5YfcEFLm3V9F07fiPSAuT4nSqZMiVjRVMmRfxm3O6gcfIJGSFTbBAQk4jeNSXss0rvADeivrJZY6Jp32wh71HCbbbWCU+bOVS1bZ3e5HNX51Ai8ucZQIizUWW1IufV51GjqVvb2QMslEhQqYKN/hel+IVSqTuouszu9CakmAyK/v/raSMypW7brPur8Xn69f6UuOjmLBGJUKE6OHZO7WPd2bpaNft613RTIk1pmzkhqq6FHyD004f3nx7Qy11lsAITLfkaWk+9SmSCv4ZQv8ZfBjR/6ouCGEQKI596IfIRSSVPESEkHhAJRjhdhXaEThCXxmhnh6CWAMIHSb7maI0DH5FUGUAZRc2AsII7SWYm+5yuB/wSSaVIFqzWEyK4ZF4LI+I0d6yIJOOuJLjmOaB3lScKLcSmmDiwK/gvn+5+drYU+pZF4v35NXe+z5oQo6dUMIcLxUZ/J8mC6jnO01h7lumAi8TpCkrkPFpepHLIKU9lPpdPCajXyauYqqW8VE7e5W7O2dWAr9fpgrOXPHXAS11FdDLk/WznIiIllePDiLpWuTPbRKvB2stKC8m8pIwnj8bhQkz1A+Js4kR08coRv5RksNaOuVJDFc7RkdjFIhlxHSDJ/3JWVXDW+4HTLlwHh+RBL8YcTb+R6eOPd+9gTpIGJ0pfQTdeJs+6Sm7jzCaHjCnSZDF7mxWNylqjS8hvW10ADuS23rdTep/BbrZdlg8Mcih0X3ZRgOTvv909fKh1fjmNaTCXrobTlbfreHY8lp4asB0tqiDDwS1Gd3kVHqx5LWFX/xeE0xBhrORlVjOXVsRmBMNlXPpjQX+zBPmPg23x5sq2r3XpYnz0OkvPeNcGPD/bvNsiwONkTO7W98QCLKHaeElKR2ytO7PNWvv3/E1y/YUh/sMDQ36P8zYbJUNi677DVCxeJXFQve30/vprJvnGY/6Hrt47Mw/1o82hvqhVHOySfe1GYVTVFUTXoprTYuFsfI9Apq5bzlY0FYhk8B0VZX/M8DBncPTz9rapNqE2d83tU3fhiQXH+NYWrswq9GKmV/ssQOOjs+GfpLba3L+jE5v04pzPde71AU7U3wTTsrhyrl52YzCFxXOJcyNr9lDYo/NzsPka2mEgOmI3X/8BUEsBAhQDFAAAAAgA1IlvXXv0JfFgBgAAWQ0AAAgAAAAAAAAAAAAAAKSBAAAAAHNldHVwLnB5UEsBAhQDFAAAAAgANYpvXWeMz9kOEAAA6SkAAAkAAAAAAAAAAAAAAKSBhgYAAGh5YnJpZC5weVBLAQIUAxQAAAAIANSJb10Rzryz6wIAAGgKAAAMAAAAAAAAAAAAAACkgbsWAABsdHgtYXBpLmpzb25QSwECFAMUAAAACADUiW9dupc26xwCAAASCAAADQAAAAAAAAAAAAAApIHQGQAAdG9vbi1hcGkuanNvblBLAQIUAxQAAAAIANSJb11HD8yimQMAANsMAAAMAAAAAAAAAAAAAACkgRccAAB3YW4tYXBpLmpzb25QSwUGAAAAAAUABQAcAQAA2h8AAAAA'))).extractall(studio)
if "google.colab" in sys.modules:
    from google.colab import files
    bundle.write_bytes(next(iter(files.upload().values())))
else:
    available=list(Path("/kaggle/input").rglob("*.zip"))
    if len(available)!=1: raise ValueError("Attach one episode bundle ZIP or build a notebook with --bundle.")
    shutil.copyfile(available[0],bundle)


In [ ]:
if USE_R2:
    names=("R2_BUCKET","R2_ENDPOINT","AWS_ACCESS_KEY_ID","AWS_SECRET_ACCESS_KEY")
    if Path("/kaggle/working").exists():
        from kaggle_secrets import UserSecretsClient
        fetch=UserSecretsClient().get_secret
    elif "google.colab" in sys.modules:
        from google.colab import userdata
        fetch=userdata.get
    else:
        fetch=os.environ.__getitem__
    for name in names: os.environ[name]=fetch(name)
    subprocess.run([sys.executable,"-m","pip","install","boto3"],check=True)
subprocess.run([sys.executable, studio/"setup.py", MODEL, "--root", root], check=True)
# For persistent output on Colab, set output to a mounted Drive directory.
subprocess.run([sys.executable, studio/"hybrid.py","gpu", bundle,output/"renders","--root",root,"--model",MODEL],check=True)


In [ ]:
import hashlib
from IPython.display import Video, display
result=output/"renders"/hashlib.sha256(bundle.read_bytes()).hexdigest()/MODEL/"episode.mp4"
display(Video(str(result),embed=True))
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(result))
